In [ ]:
import sys

!{sys.executable} -m pip install -q \
    evaluate \
    transformers==4.46.3 \
    datasets==3.1.0 \
    accelerate==1.1.1 \
    huggingface_hub

In [ ]:
!{sys.executable} -m pip show evaluate

In [ ]:
import transformers
import torch
import accelerate

print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("accelerate:", accelerate.__version__)

print("GPUs:", torch.cuda.device_count())
print("CUDA:", torch.cuda.is_available())

In [ ]:
import sys
print(sys.executable)

!pip show evaluate

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import classification_report

# ==========================================
# --- CẤU HÌNH ---
# ==========================================
os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_checkpoint = "Akiya-Vyre/deberta-v3-fever"

# ==========================================
# --- BƯỚC 1: CHUẨN BỊ KHO CANDIDATE ---
# ==========================================
print("[INFO] Đang tải dữ liệu và chuẩn bị candidate pool...")
dataset = load_dataset("pietrolesci/nli_fever")
sim_model = SentenceTransformer("all-MiniLM-L6-v2").to(device)
candidate_pool = dataset["train"]["premise"][:10000]
pool_emb = sim_model.encode(candidate_pool, convert_to_tensor=True, show_progress_bar=True)

# ==========================================
# --- BƯỚC 2: HÀM AUGMENT DỮ LIỆU (CHỈ NEI) ---
# ==========================================
def build_hnm_dataset(dataset, ratio=0.10):
    new_data = {"hypothesis": [], "premise": [], "label": []}
    
    for sample in dataset:
        hyp, prem, label = sample["hypothesis"], sample["premise"], sample["label"]
        
        # Giữ sample gốc
        new_data["hypothesis"].append(hyp)
        new_data["premise"].append(prem)
        new_data["label"].append(label)

        # Chỉ Augment nhãn 2 (NEI)
        if label == 2 and random.random() < ratio:
            hyp_emb = sim_model.encode(hyp, convert_to_tensor=True)
            scores = util.cos_sim(hyp_emb, pool_emb)[0]
            
            # Lọc vùng xám 0.3 - 0.5 (Related but not entail)
            valid_idx = torch.where((scores > 0.30) & (scores < 0.50))[0]
            if len(valid_idx) > 0:
                idx = random.choice(valid_idx.tolist())
                fake_prem = candidate_pool[idx]
                if fake_prem != prem: # Tránh trùng gold
                    new_data["hypothesis"].append(hyp)
                    new_data["premise"].append(fake_prem)
                    new_data["label"].append(2) # Vẫn là NEI
    return new_data

# ==========================================
# --- BƯỚC 3: TẠO TRAIN SET VÀ PREPROCESS ---
# ==========================================
print("[INFO] Đang augment và tokenize dữ liệu...")
hnm_train = Dataset.from_dict(build_hnm_dataset(dataset["train"]))
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess(examples):
    result = tokenizer(examples["hypothesis"], examples["premise"], truncation=True, max_length=256)
    result["labels"] = examples["label"]
    if "token_type_ids" in result: 
        del result["token_type_ids"]
    return result

tokenized_train = hnm_train.map(preprocess, batched=True, remove_columns=["hypothesis", "premise", "label"])
tokenized_dev = dataset["dev"].map(preprocess, batched=True, remove_columns=["hypothesis", "premise", "label"])

# ==========================================
# --- BƯỚC 4: CUSTOM FOCAL LOSS TRAINER ---
# ==========================================
class FocalLossTrainer(Trainer):
    def __init__(self, *args, alpha=0.25, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha
        self.gamma = gamma

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # HuggingFace version mới có thể truyền thêm num_items_in_batch qua kwargs, ta dùng **kwargs để hứng hết
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Tính Cross Entropy cơ bản (không reduction để tính riêng từng sample)
        ce_loss = F.cross_entropy(logits, labels, reduction="none")
        
        # Tính xác suất pt của class đúng
        pt = torch.exp(-ce_loss)
        
        # Công thức Focal Loss
        focal_loss = (self.alpha * (1 - pt) ** self.gamma * ce_loss).mean()
        
        return (focal_loss, outputs) if return_outputs else focal_loss

# ==========================================
# --- BƯỚC 5: KHỞI TẠO MODEL & TRAIN ---
# ==========================================
print("[INFO] Khởi tạo model và bắt đầu training...")
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3)

# Tái xác nhận mapping
model.config.id2label = {0: "SUPPORTS", 1: "REFUTES", 2: "NOT_ENOUGH_INFO"}
model.config.label2id = {"SUPPORTS": 0, "REFUTES": 1, "NOT_ENOUGH_INFO": 2}
model.to(device)

training_args = TrainingArguments(
    output_dir="deberta-v3-fever-focal-loss",
    learning_rate=8e-6,
    num_train_epochs=1,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=False, # CỰC KỲ QUAN TRỌNG NHƯ ANH ĐÃ NOTE: Tránh NaN
    report_to="none"
)

# SỬ DỤNG FOCAL LOSS TRAINER THAY VÌ TRAINER GỐC
trainer = FocalLossTrainer(
    model=model, 
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer)
)

trainer.train()
trainer.save_model("./final_model_focal_loss")

# ==========================================
# --- BƯỚC 6: ĐÁNH GIÁ (EVALUATION) ---
# ==========================================
print("[INFO] Đang đánh giá model...")
eval_result = trainer.evaluate()
print("Kết quả Evaluation cuối cùng:", eval_result)

preds = trainer.predict(tokenized_dev)
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print("\n================ CLASSIFICATION REPORT ================")
print(classification_report(
    y_true, 
    y_pred, 
    target_names=["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO"]
))

In [ ]:
from sklearn.metrics import classification_report

eval_result = trainer.evaluate()
print("Kết quả Evaluation cuối cùng:", eval_result)

# Trích xuất phân phối nhãn dự đoán thực tế
preds = trainer.predict(tokenized_dev) # Sử dụng biến đã định nghĩa ở Block 1
y_pred = np.argmax(preds.predictions, axis=1)
y_true = preds.label_ids

print("\n================ CLASSIFICATION REPORT ================")
print(classification_report(
    y_true, 
    y_pred, 
    target_names=["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO"]
))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score
)
from scipy.special import softmax

# logits -> probability
probs = softmax(
    preds.predictions,
    axis=1
)

y_true = preds.label_ids

# one-hot labels
y_bin = label_binarize(
    y_true,
    classes=[0, 1, 2]
)

class_names = [
    "SUPPORTS",
    "REFUTES",
    "NEI"
]

plt.figure(figsize=(8, 6))

for i in range(3):

    precision, recall, _ = precision_recall_curve(
        y_bin[:, i],
        probs[:, i]
    )

    ap = average_precision_score(
        y_bin[:, i],
        probs[:, i]
    )

    plt.plot(
        recall,
        precision,
        label=f"{class_names[i]} (AP={ap:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

print("[HUGGINGFACE] Bắt đầu quy trình kết nối và đẩy mô hình...")

# 1. Lấy Token bảo mật từ Kaggle Secrets
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    
    # 2. Định nghĩa Repo đích nâng cấp chống nhiễu của bạn
    target_repo_id = "Akiya-Vyre/deberta-v3-fever" # Hoặc "Akiya-Vyre/deberta-v3-fever-hnm" tùy bạn chọn
    
    # Push lần lượt Model và Tokenizer lên Hub
    model.push_to_hub(target_repo_id, safe_serialization=True)
    tokenizer.push_to_hub(target_repo_id)
    
    print(f"[SUCCESS] Đã đồng bộ thành công Model nâng cấp lên Hub: https://huggingface.co/{target_repo_id}")

except Exception as e:
    print(f"[ERROR] Quá trình upload thất bại do: {str(e)}")
    print("Lưu ý: Bạn vẫn có thể tải mô hình từ thư mục cục bộ `./final_model_hnm` vừa lưu ở Block 2.")